In [8]:
# run_notears_package_linear.py
# ------------------------------------------------------------
# NOTEARS (package version) wrapper script
# - Uses: from notears.linear import notears_linear
# - Preprocess: drop targets + numeric-only + NaN median + StandardScaler
# - Save: edges/adj/graphml/gexf/json
#
# Requirements:
#   pip install numpy pandas scikit-learn networkx notears
# ------------------------------------------------------------

import os
import json
import warnings
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import networkx as nx

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")


# =========================
# Config
# =========================
DATA_PATH = "./training_data_normalized.csv"   # 원하는 입력 파일로 바꾸세요
OUT_DIR = "./dag_out/NOTEARS"

RANDOM_STATE = 42
MAX_FEATURES = None

TARGET_CANDIDATES = ["label", "target", "y", "failure", "bank_failure", "default", "is_failed"]

# NOTEARS hyperparams (package)
LAMBDA1 = 0.005            # sparsity
LOSS_TYPE = "l2"         # continuous data: 'l2'

# post-processing
W_THRESHOLD = 0.0        # 0.05~0.3 등으로 실험 가능 (abs(W)<thr -> 0)
ENFORCE_DAG_SAFETY = True  # cycle이 남으면 제거하는 안전장치(보통은 False여도 됨)


# =========================
# Data loading
# =========================
def load_numeric_X(
    data_path: str,
    drop_target_candidates: bool = True,
    max_features: Optional[int] = None,
    random_state: int = 42,
    standardize: bool = True
) -> Tuple[pd.DataFrame, np.ndarray, List[str]]:
    np.random.seed(random_state)
    df = pd.read_csv(data_path, low_memory=False)

    # 흔한 인덱스 컬럼 제거(있으면)
    for c in ["Unnamed: 0", "index", "__index_level_0__"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    if drop_target_candidates:
        # 대소문자/공백 변형 흡수
        norm_cols = {c.strip().lower(): c for c in df.columns}
        drop_cols = []
        for tc in TARGET_CANDIDATES:
            if tc in norm_cols:
                drop_cols.append(norm_cols[tc])
        if drop_cols:
            print(f"[INFO] drop target candidates: {drop_cols}")
            df = df.drop(columns=drop_cols)

    # numeric only (object/categorical 컬럼이 섞이면 패키지 입력에서 깨지기 쉬움)
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    df = df[num_cols].copy()

    if max_features is not None and df.shape[1] > max_features:
        df = df.iloc[:, :max_features].copy()
        print(f"[INFO] feature capped: {max_features}")

    # NaN -> median
    for c in df.columns:
        if df[c].isna().any():
            df[c] = df[c].fillna(df[c].median())

    col_names = df.columns.tolist()
    X = df.values.astype(float)

    if standardize:
        X = StandardScaler().fit_transform(X)

    print(f"[INFO] X shape: {X.shape}")
    return df, X, col_names


# =========================
# Cycle-break (optional safety)
# =========================
def break_cycles_by_removing_small_edges(W: np.ndarray) -> np.ndarray:
    """
    그래프에 cycle이 남아있으면, cycle에 포함된 edge 중 |weight|가 가장 작은 edge를 제거 반복.
    """
    W2 = W.copy()
    d = W2.shape[0]

    def build_adj():
        G = {i: [] for i in range(d)}
        for i in range(d):
            for j in range(d):
                if i != j and abs(W2[i, j]) > 0:
                    G[i].append(j)
        return G

    def find_cycle_edges(G):
        color = [0] * d
        parent = [-1] * d

        def dfs(u):
            color[u] = 1
            for v in G[u]:
                if color[v] == 0:
                    parent[v] = u
                    cyc = dfs(v)
                    if cyc is not None:
                        return cyc
                elif color[v] == 1:
                    # back-edge => cycle
                    nodes = [v]
                    cur = u
                    while cur != v and cur != -1:
                        nodes.append(cur)
                        cur = parent[cur]
                    nodes.append(v)
                    nodes = nodes[::-1]
                    return [(a, b) for a, b in zip(nodes[:-1], nodes[1:])]
            color[u] = 2
            return None

        for s in range(d):
            if color[s] == 0:
                cyc = dfs(s)
                if cyc is not None:
                    return cyc
        return None

    removed = 0
    while True:
        cyc = find_cycle_edges(build_adj())
        if cyc is None:
            break
        mags = [(abs(W2[i, j]), i, j) for (i, j) in cyc]
        mags.sort(key=lambda x: x[0])
        _, i_min, j_min = mags[0]
        W2[i_min, j_min] = 0.0
        removed += 1

    if removed:
        print(f"[INFO] cycle-break removed edges: {removed}")

    return W2


def is_dag(W: np.ndarray) -> bool:
    d = W.shape[0]
    G = nx.DiGraph()
    G.add_nodes_from(range(d))
    for i in range(d):
        for j in range(d):
            if i != j and abs(W[i, j]) > 0:
                G.add_edge(i, j)
    return nx.is_directed_acyclic_graph(G)


# =========================
# Save artifacts
# =========================
def save_artifacts(W: np.ndarray, col_names: List[str], out_dir: str, alg_name: str) -> None:
    os.makedirs(out_dir, exist_ok=True)

    edges = []
    for i, src in enumerate(col_names):
        for j, tgt in enumerate(col_names):
            if i != j and abs(W[i, j]) > 0:
                edges.append([src, tgt, float(W[i, j])])

    edge_df = pd.DataFrame(edges, columns=["source", "target", "weight"])
    edge_path = os.path.join(out_dir, f"edges_{alg_name}.csv")
    edge_df.to_csv(edge_path, index=False)

    adj_df = pd.DataFrame(W, index=col_names, columns=col_names)
    adj_path = os.path.join(out_dir, f"adj_{alg_name}.csv")
    adj_df.to_csv(adj_path)

    G = nx.DiGraph()
    for n in col_names:
        G.add_node(n)
    for _, r in edge_df.iterrows():
        G.add_edge(r["source"], r["target"], weight=float(r["weight"]))

    graphml_path = os.path.join(out_dir, f"graph_{alg_name}.graphml")
    gexf_path = os.path.join(out_dir, f"graph_{alg_name}.gexf")
    nx.write_graphml(G, graphml_path)
    nx.write_gexf(G, gexf_path)

    nodes = [{"id": n} for n in G.nodes()]
    jedges = [{"source": u, "target": v, "weight": float(G[u][v].get("weight", 0.0))} for u, v in G.edges()]
    json_path = os.path.join(out_dir, f"graph_{alg_name}.json")
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"nodes": nodes, "edges": jedges}, f, ensure_ascii=False, indent=2)

    print(f"[SAVE] {alg_name}")
    print(f"  - {edge_path} (n_edges={len(edge_df)})")
    print(f"  - {adj_path}")
    print(f"  - {graphml_path}")
    print(f"  - {gexf_path}")
    print(f"  - {json_path}")


# =========================
# Main
# =========================
def main():
    _, X, col_names = load_numeric_X(
        DATA_PATH,
        drop_target_candidates=True,
        max_features=MAX_FEATURES,
        random_state=RANDOM_STATE,
        standardize=True  # package 예제에 맞춤
    )

    print("[RUN] NOTEARS-linear (package: notears.linear.notears_linear)")
    try:
        from notears.linear import notears_linear
    except Exception as e:
        print("[ERROR] cannot import notears. Install: pip install notears")
        raise e

    W = notears_linear(X, lambda1=LAMBDA1, loss_type=LOSS_TYPE)
    np.fill_diagonal(W, 0.0)

    # thresholding
    if W_THRESHOLD and W_THRESHOLD > 0:
        W[np.abs(W) < W_THRESHOLD] = 0.0
        np.fill_diagonal(W, 0.0)

    # optional DAG safety
    if ENFORCE_DAG_SAFETY:
        if not is_dag(W):
            print("[WARN] W is not a DAG (numerical/threshold issues). Running cycle-break safety.")
            W = break_cycles_by_removing_small_edges(W)
        else:
            print("[INFO] DAG check: OK")

    save_artifacts(W, col_names, OUT_DIR, "NOTEARS")
    print("[DONE] NOTEARS(PACKAGE) artifacts saved.")


if __name__ == "__main__":
    main()


[INFO] drop target candidates: ['label']
[INFO] X shape: (17881, 13)
[RUN] NOTEARS-linear (package: notears.linear.notears_linear)
[INFO] DAG check: OK
[SAVE] NOTEARS
  - ./dag_out/NOTEARS\edges_NOTEARS.csv (n_edges=18)
  - ./dag_out/NOTEARS\adj_NOTEARS.csv
  - ./dag_out/NOTEARS\graph_NOTEARS.graphml
  - ./dag_out/NOTEARS\graph_NOTEARS.gexf
  - ./dag_out/NOTEARS\graph_NOTEARS.json
[DONE] NOTEARS(PACKAGE) artifacts saved.
